# Swin Transformer-Based Multi-Class Skin Cancer Diagnosis
### Complete Jupyter Notebook Project using the HAM10000 Dataset

This notebook implements an end-to-end deep learning pipeline for **7-class skin lesion classification** using a **Swin Transformer** backbone on the **Skin Cancer MNIST: HAM10000** dataset.

## Project Summary
**Dataset:** HAM10000 (≈10,015 dermoscopic images)  
**Classes:** `nv`, `mel`, `bcc`, `akiec`, `bkl`, `df`, `vasc`  
**Core preprocessing:** image resizing, normalization, class balancing with `RandomOverSampler()`  
**Model:** Swin Transformer  
**Task:** Multi-class skin cancer image classification

## What this notebook includes
- Reproducible project setup
- Data loading and metadata preparation
- Exploratory data analysis
- Stratified train/validation/test split
- Class balancing with `RandomOverSampler`
- PyTorch `Dataset` and `DataLoader`
- Swin Transformer training pipeline
- Validation and test evaluation
- Curves for loss and accuracy
- Confusion matrix and classification report
- Model checkpoint saving
- Single-image inference utility
- Clean project structure and best practices

> **Note:** This notebook is designed as a full academic/project submission template.  
> You only need to set the correct dataset paths before running.


## Step-by-Step Project Roadmap

A complete project should include the following components:

### 1. Problem Definition
Define the classification objective clearly:
- Input: dermoscopic skin lesion image
- Output: one of seven lesion classes
- Goal: assist automated skin lesion diagnosis

### 2. Dataset Understanding
Document:
- source of the dataset
- class names
- number of images
- metadata fields
- class imbalance

### 3. Data Preprocessing
Include:
- missing value checks
- class distribution analysis
- label encoding
- stratified splitting
- resizing
- normalization
- class balancing strategy

### 4. Model Design
Explain:
- why Swin Transformer is suitable
- transfer learning strategy
- loss function
- optimizer
- scheduler
- regularization

### 5. Training Pipeline
A professional project should include:
- seed fixing
- mixed precision (if available)
- early stopping
- checkpoint saving
- logging of metrics per epoch

### 6. Evaluation
Include:
- training loss
- validation loss
- training accuracy
- test loss
- test accuracy
- confusion matrix
- precision / recall / F1-score

### 7. Interpretation and Discussion
Discuss:
- which classes are confused
- effects of imbalance
- limitations of the model
- possible clinical risks of misclassification

### 8. Reproducibility and Deployment Readiness
A strong project also includes:
- saved model weights
- inference function
- class mapping
- environment requirements
- clean code structure

This notebook follows all of the above.


In [ ]:
# ============================================================
# 1. Environment Setup
# ============================================================
# Run this cell once if the required libraries are not installed.

# Uncomment when needed:
# !pip install -q timm albumentations imbalanced-learn seaborn scikit-learn kaggle



In [ ]:
# ============================================================
# 2. Imports and Global Configuration
# ============================================================
import os
import gc
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm

warnings.filterwarnings("ignore")

# Optional: nicer plots if seaborn is available
try:
    import seaborn as sns
    sns.set_context("notebook")
    sns.set_style("whitegrid")
except Exception:
    pass

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = "swin_tiny_patch4_window7_224"
CHECKPOINT_PATH = "best_swin_ham10000.pth"

print(f"Using device: {DEVICE}")


## Dataset Folder Structure

This notebook assumes a Kaggle-style structure similar to:

```text
HAM10000/
├── HAM10000_metadata.csv
├── ham10000_images_part_1/
│   ├── ISIC_0024306.jpg
│   └── ...
└── ham10000_images_part_2/
    ├── ISIC_0024307.jpg
    └── ...
```

Update the paths below to match your environment.


In [ ]:
# ============================================================
# 3. Dataset Paths
# ============================================================
DATA_ROOT = Path("./HAM10000")  # <-- change this
METADATA_PATH = DATA_ROOT / "HAM10000_metadata.csv"
IMAGE_DIRS = [
    DATA_ROOT / "ham10000_images_part_1",
    DATA_ROOT / "ham10000_images_part_2",
]

assert METADATA_PATH.exists(), f"Metadata file not found: {METADATA_PATH}"
for folder in IMAGE_DIRS:
    assert folder.exists(), f"Image folder not found: {folder}"

print("Paths look good.")


In [ ]:
# ============================================================
# 4. Load Metadata and Build Image Paths
# ============================================================
df = pd.read_csv(METADATA_PATH)
print("Metadata shape:", df.shape)
display(df.head())

def find_image_path(image_id, image_dirs):
    for d in image_dirs:
        path = d / f"{image_id}.jpg"
        if path.exists():
            return str(path)
    return None

df["image_path"] = df["image_id"].apply(lambda x: find_image_path(x, IMAGE_DIRS))
df = df[df["image_path"].notna()].reset_index(drop=True)

print("Usable rows after path matching:", len(df))
print("Null image paths:", df["image_path"].isna().sum())


In [ ]:
# ============================================================
# 5. Basic EDA
# ============================================================
CLASS_NAMES = {
    "nv": "Melanocytic nevi",
    "mel": "Melanoma",
    "bcc": "Basal cell carcinoma",
    "akiec": "Actinic keratoses",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "vasc": "Vascular lesions",
}

df["class_name"] = df["dx"].map(CLASS_NAMES)

print("Class distribution:")
display(df["dx"].value_counts())

plt.figure(figsize=(10, 5))
df["dx"].value_counts().plot(kind="bar")
plt.title("Class Distribution in HAM10000")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

print("\nMissing values:")
display(df.isnull().sum())


In [ ]:
# ============================================================
# 6. Visualize Sample Images
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
axes = axes.flatten()

sample_df = df.groupby("dx", group_keys=False).apply(lambda x: x.sample(1, random_state=42))
sample_df = sample_df.reset_index(drop=True)

for idx, ax in enumerate(axes):
    if idx < len(sample_df):
        row = sample_df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(f'{row["dx"]}\n{CLASS_NAMES[row["dx"]]}', fontsize=10)
        ax.axis("off")
    else:
        ax.axis("off")

plt.suptitle("One Example Image per Class", fontsize=14)
plt.tight_layout()
plt.show()


## Label Encoding and Split Strategy

We use:
- **Label encoding** for the target
- **Stratified splitting** to preserve class proportions
- **RandomOverSampler** on the training set only to reduce imbalance leakage


In [ ]:
# ============================================================
# 7. Label Encoding and Train/Val/Test Split
# ============================================================
le = LabelEncoder()
df["label"] = le.fit_transform(df["dx"])

label_to_class = {i: cls for i, cls in enumerate(le.classes_)}
class_to_label = {cls: i for i, cls in label_to_class.items()}

print("Encoded classes:", label_to_class)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42,
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain class distribution (before oversampling):")
display(train_df["dx"].value_counts())


In [ ]:
# ============================================================
# 8. Class Balancing with RandomOverSampler
# ============================================================
ros = RandomOverSampler(random_state=42)

X_resampled, y_resampled = ros.fit_resample(
    train_df[["image_path"]],
    train_df["label"]
)

train_balanced_df = pd.DataFrame({
    "image_path": X_resampled["image_path"],
    "label": y_resampled
})

train_balanced_df["dx"] = train_balanced_df["label"].map(label_to_class)
train_balanced_df["class_name"] = train_balanced_df["dx"].map(CLASS_NAMES)

print("Balanced training class distribution:")
display(train_balanced_df["dx"].value_counts())

plt.figure(figsize=(10, 5))
train_balanced_df["dx"].value_counts().plot(kind="bar")
plt.title("Balanced Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# ============================================================
# 9. Image Transforms
# ============================================================
# We use torchvision transforms for simplicity and portability.
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


In [ ]:
# ============================================================
# 10. Custom Dataset
# ============================================================
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = HAM10000Dataset(train_balanced_df, transform=train_transforms)
val_dataset = HAM10000Dataset(val_df, transform=eval_transforms)
test_dataset = HAM10000Dataset(test_df, transform=eval_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


## Why Swin Transformer?

Swin Transformer is a strong fit for dermoscopic image classification because it:
- captures both local and global visual patterns,
- builds **hierarchical features** across stages,
- scales well to image recognition tasks,
- often performs strongly with transfer learning.

In this project, we fine-tune a pretrained Swin Transformer and replace its classifier head for **7-class output**.


In [ ]:
# ============================================================
# 11. Build the Swin Transformer Model
# ============================================================
NUM_CLASSES = len(label_to_class)

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    verbose=True
)

print(model.__class__.__name__)


In [ ]:
# ============================================================
# 12. Training and Evaluation Utilities
# ============================================================
from contextlib import nullcontext

def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()

def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    epoch_loss = 0.0
    epoch_acc = 0.0
    all_preds = []
    all_labels = []

    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    context = torch.enable_grad if is_train else torch.no_grad
    with context():
        for images, labels in tqdm(loader, leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            amp_context = torch.cuda.amp.autocast if use_amp else nullcontext
            with amp_context():
                outputs = model(images)
                loss = criterion(outputs, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            acc = accuracy_from_logits(outputs, labels)

            epoch_loss += loss.item() * images.size(0)
            epoch_acc += acc * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss /= len(loader.dataset)
    epoch_acc /= len(loader.dataset)

    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)

class EarlyStopping:
    def __init__(self, patience=4, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

early_stopper = EarlyStopping(patience=4, min_delta=1e-4)


In [ ]:
# ============================================================
# 13. Model Training
# ============================================================
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

best_val_loss = float("inf")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")

    train_loss, train_acc, _, _ = run_one_epoch(
        model, train_loader, criterion, optimizer=optimizer
    )
    val_loss, val_acc, _, _ = run_one_epoch(
        model, val_loader, criterion, optimizer=None
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"✅ Best model saved to {CHECKPOINT_PATH}")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print("⏹ Early stopping triggered.")
        break

    print("-" * 60)

elapsed = time.time() - start_time
print(f"Training completed in {elapsed / 60:.2f} minutes.")


In [ ]:
# ============================================================
# 14. Training Curves
# ============================================================
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))
plt.plot(epochs, history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, history["val_loss"], marker="o", label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(epochs, history["train_acc"], marker="o", label="Train Accuracy")
plt.plot(epochs, history["val_acc"], marker="o", label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
# ============================================================
# 15. Test Set Evaluation
# ============================================================
best_model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES
).to(DEVICE)

best_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
best_model.eval()

test_loss, test_acc, test_preds, test_labels = run_one_epoch(
    best_model, test_loader, criterion, optimizer=None
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


In [ ]:
# ============================================================
# 16. Detailed Metrics
# ============================================================
target_names = [CLASS_NAMES[c] for c in le.classes_]

print("Classification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=target_names,
    digits=4
))

cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# ============================================================
# 17. Single Image Inference Utility
# ============================================================
def predict_image(image_path, model, transform, label_encoder):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))
        pred_class = label_encoder.inverse_transform([pred_idx])[0]

    return {
        "predicted_label_index": pred_idx,
        "predicted_class_code": pred_class,
        "predicted_class_name": CLASS_NAMES[pred_class],
        "confidence": float(probs[pred_idx]),
        "all_probabilities": {
            label_encoder.inverse_transform([i])[0]: float(probs[i])
            for i in range(len(probs))
        }
    }

# Example:
# sample_path = test_df.iloc[0]["image_path"]
# result = predict_image(sample_path, best_model, eval_transforms, le)
# print(result)


In [ ]:
# ============================================================
# 18. Visual Demo on a Test Image
# ============================================================
idx = 0
demo_path = test_df.iloc[idx]["image_path"]
demo_true = test_df.iloc[idx]["dx"]

result = predict_image(demo_path, best_model, eval_transforms, le)

img = Image.open(demo_path).convert("RGB")
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title(
    f"True: {demo_true} ({CLASS_NAMES[demo_true]})\n"
    f"Pred: {result['predicted_class_code']} ({result['predicted_class_name']})\n"
    f"Confidence: {result['confidence']:.4f}"
)
plt.show()


In [ ]:
# ============================================================
# 19. Save Project Artifacts
# ============================================================
artifacts = {
    "class_to_label": class_to_label,
    "label_to_class": label_to_class,
    "class_name_mapping": CLASS_NAMES,
    "history": history,
    "model_name": MODEL_NAME,
    "img_size": IMG_SIZE,
    "checkpoint_path": CHECKPOINT_PATH
}

import json
with open("project_artifacts.json", "w") as f:
    json.dump(artifacts, f, indent=4)

print("Saved:")
print("- Model checkpoint:", CHECKPOINT_PATH)
print("- Project metadata: project_artifacts.json")


## Expected Outcome

This project aims to produce a **high-performing Swin Transformer-based classifier** for automated multi-class skin lesion recognition.

### Deliverables produced by this notebook
- trained Swin Transformer model
- loss and accuracy curves
- validation and test metrics
- confusion matrix
- reusable inference function
- saved project artifacts

### Possible Extensions
To make the project even stronger, you can add:
- weighted loss instead of or alongside oversampling
- test-time augmentation
- Grad-CAM or attention visualization
- k-fold cross-validation
- hyperparameter tuning
- deployment with Streamlit or Gradio
- explainability section for academic presentation

## Important Limitation
This notebook builds a research-grade classification pipeline, but **it is not a clinical diagnostic system**. Predictions should never replace expert medical judgment.


## Suggested Final Project Report Sections

For a complete academic/project submission, structure your written report as:

1. **Introduction**
2. **Problem Statement**
3. **Dataset Description**
4. **Preprocessing**
5. **Model Architecture**
6. **Training Strategy**
7. **Evaluation Metrics**
8. **Results**
9. **Discussion**
10. **Limitations**
11. **Future Work**
12. **Conclusion**

This notebook already provides the implementation backbone for all of those sections.
